# 02 — Data Cleaning & Validation

**Bluestock Mutual Fund Analytics Capstone**

This notebook cleans and validates the project's SQLite datasets without modifying the source database. It standardizes column names, text, dates and numeric fields, checks duplicates and missing values, validates key fields, checks date coverage, and exports cleaned CSVs to `data/processed/`.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display

# Works when launched from the project root or from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DB_PATH = PROJECT_ROOT / "bluestock_mf.db"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Processed directory:", PROCESSED_DIR)


## 1. Inventory the SQLite database

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
        conn
    )

print(f"Tables found: {len(tables)}")
display(tables)


## 2. Load every table

In [ ]:
datasets = {}

with sqlite3.connect(DB_PATH) as conn:
    for table in tables["name"]:
        datasets[table] = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)

inventory = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum())
    }
    for name, df in datasets.items()
])

display(inventory)


## 3. Cleaning functions

In [ ]:
def clean_column_names(df):
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return out


def standardize_text(df):
    out = df.copy()
    for col in out.select_dtypes(include=["object", "string"]).columns:
        out[col] = out[col].astype("string").str.strip()
        out[col] = out[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return out


def standardize_dates(df):
    out = df.copy()
    for col in out.columns:
        name = col.lower()
        if name == "date" or name.endswith("_date") or name == "month" or "date" in name:
            converted = pd.to_datetime(out[col], errors="coerce")
            if converted.notna().any():
                out[col] = converted
    return out


def coerce_numeric_columns(df):
    out = df.copy()
    hints = (
        "amount", "aum", "nav", "return", "cagr", "sharpe", "std",
        "volatility", "beta", "alpha", "drawdown", "expense", "ratio",
        "inflow", "outflow", "folio", "count", "age", "value", "close",
        "open", "high", "low", "price", "weight", "percent", "pct"
    )
    for col in out.columns:
        if any(h in col.lower() for h in hints):
            if out[col].dtype == "object" or str(out[col].dtype).startswith("string"):
                cleaned = (
                    out[col].astype("string")
                    .str.replace(",", "", regex=False)
                    .str.replace("%", "", regex=False)
                )
                converted = pd.to_numeric(cleaned, errors="coerce")
                if converted.notna().any():
                    out[col] = converted
    return out


def clean_table(df):
    out = clean_column_names(df)
    out = standardize_text(out)
    out = standardize_dates(out)
    out = coerce_numeric_columns(out)
    out = out.drop_duplicates().reset_index(drop=True)
    return out


## 4. Apply cleaning and compare before/after

In [ ]:
cleaned = {name: clean_table(df) for name, df in datasets.items()}

clean_inventory = pd.DataFrame([
    {
        "table": name,
        "rows_before": len(datasets[name]),
        "rows_after": len(df),
        "duplicates_removed": len(datasets[name]) - len(df),
        "missing_cells_after": int(df.isna().sum().sum())
    }
    for name, df in cleaned.items()
])

display(clean_inventory)


## 5. Validate important keys and columns

In [ ]:
expected_keys = {
    "dim_fund": ["amfi_code"],
    "fact_nav": ["amfi_code", "date"],
    "fact_performance": ["amfi_code"],
    "fact_transactions": ["date"],
}

checks = []

for table, required in expected_keys.items():
    if table not in cleaned:
        checks.append({
            "table": table,
            "status": "MISSING TABLE",
            "missing_columns": ", ".join(required)
        })
        continue

    missing = [c for c in required if c not in cleaned[table].columns]
    checks.append({
        "table": table,
        "status": "PASS" if not missing else "MISSING COLUMN",
        "missing_columns": ", ".join(missing)
    })

display(pd.DataFrame(checks))


## 6. Missing-value report

In [ ]:
missing_rows = []

for table, df in cleaned.items():
    for col in df.columns:
        n = int(df[col].isna().sum())
        if n:
            missing_rows.append({
                "table": table,
                "column": col,
                "missing_count": n,
                "missing_pct": round(n / len(df) * 100, 2) if len(df) else 0
            })

missing_report = pd.DataFrame(missing_rows)

if missing_report.empty:
    print("No missing values found after cleaning.")
else:
    display(missing_report.sort_values(["table", "missing_pct"], ascending=[True, False]))


## 7. Duplicate and uniqueness checks

In [ ]:
key_results = []

if "dim_fund" in cleaned and "amfi_code" in cleaned["dim_fund"].columns:
    df = cleaned["dim_fund"]
    null_keys = int(df["amfi_code"].isna().sum())
    duplicate_keys = int(df.duplicated(["amfi_code"]).sum())

    key_results.append({
        "table": "dim_fund",
        "key": "amfi_code",
        "null_keys": null_keys,
        "duplicate_keys": duplicate_keys,
        "status": "PASS" if null_keys == 0 and duplicate_keys == 0 else "REVIEW"
    })

display(pd.DataFrame(key_results))


## 8. Date coverage checks

In [ ]:
def date_report(df, date_col="date"):
    if date_col not in df.columns or df.empty:
        return None

    s = pd.to_datetime(df[date_col], errors="coerce").dropna()
    if s.empty:
        return None

    return {
        "min_date": s.min().date(),
        "max_date": s.max().date(),
        "unique_dates": s.dt.normalize().nunique()
    }

reports = []

for table in ["fact_nav", "fact_benchmark", "fact_transactions"]:
    if table in cleaned:
        result = date_report(cleaned[table])
        if result:
            result["table"] = table
            reports.append(result)

display(pd.DataFrame(reports))


## 9. Export cleaned CSVs

In [ ]:
exported = []

for name, df in cleaned.items():
    output_path = PROCESSED_DIR / f"{name}.csv"
    export_df = df.copy()

    for col in export_df.columns:
        if pd.api.types.is_datetime64_any_dtype(export_df[col]):
            export_df[col] = export_df[col].dt.strftime("%Y-%m-%d")

    export_df.to_csv(output_path, index=False)

    exported.append({
        "table": name,
        "file": str(output_path),
        "rows": len(export_df)
    })

display(pd.DataFrame(exported))


## 10. Final cleaning checklist

- Uses `pathlib.Path`; no hard-coded absolute Windows paths.
- Does not modify the source SQLite database.
- Standardizes column names and text values.
- Converts date-like fields safely with `errors="coerce"`.
- Converts numeric-looking fields conservatively.
- Removes exact duplicate rows from exported datasets.
- Reports missing values instead of silently imputing them.
- Validates key columns such as `amfi_code`.
- Checks date coverage for NAV, benchmark and transactions.
- Exports cleaned datasets to `data/processed/`.

**Trading-day note:** weekends and market holidays are not automatically treated as missing observations. Performance analytics should explicitly handle the trading calendar and use forward-fill only where the analytical requirement calls for a complete date index.
